In [4]:
import torch
import torch.nn as nn
# summary包用于便捷打印模型的结构和参数量等信息
from torchsummary import summary

# 搭建简单神经网络
PyTorch提供了 `nn.Module` 类来搭建神经网络. 可以通过继承 `nn.Module` 类来定义自己的神经网络模型.

- 定义一个神经网络类, 继承自 `nn.Module` 类.
- 在 `__init__` 方法中定义网络的层.
- 在 `forward` 方法中定义前向传播的计算过程.

    ```python
    # =============================================
    import torch
    import torch.nn as nn

    class SimpleNet(nn.Module):
        def __init__(self):
            super(SimpleNet, self).__init__()
            self.layer1 = nn.Linear(10, 20)  # 输入维度为10, 输出维度为20
            self.layer2 = nn.Linear(20, 10)  # 输入维度为20, 输出维度为10
            self.layer3 = nn.Linear(10, 1)   # 输入维度为10, 输出维度为1

        def forward(self, x):
            x = torch.relu(self.layer1(x))     # 第一层使用ReLU激活函数
            x = torch.relu(self.layer2(x))     # 第二层使用ReLU激活函数
            x = torch.sigmoid(self.layer3(x))  # 第三层使用Sigmoid激活函数 (假设是二分类问题)
            return x
    # =============================================
    ```

<div align="center">
    <img src="./assets/nn.png" alt="神经网络结构图"/>
</div>

下面例子为上图中神经网络的实现. 其中的 `+1` 代表了偏置项参数 `b` 的存在, 也就是每一层的未激活原始输出是 $ wx + 1 \cdot b = wx + b $, 输入维度大小不包括这个 `+1`

In [5]:
# 自定义的神经网络要继承自 nn.Module 类
class SimpleNet(nn.Module):
    def __init__(self):
        # 先初始化父类成员. 📌📌与Java不同, Python中父类的初始化方法需要显式调用, 否则父类的成员变量不会被初始化.
        super().__init__()
        # 定义网络的层
        # 第一隐藏层, 输入维度为3 (输入特征数量为3), 输出维度为3
        self.layer1 = nn.Linear(3, 3)
        # 第二隐藏层, 输入3, 输出2
        self.layer2 = nn.Linear(3, 2)
        # 输出层, 输入2, 输出2
        self.layer3 = nn.Linear(2, 2)

        # (可选)手动初始化权重参数
        # 第一层使用标准化(正态分布)的Xavier初始化权重参数
        nn.init.xavier_normal_(self.layer1.weight)
        nn.init.zeros_(self.layer1.bias) # 初始化偏置为0
        # 第二层使用标准化(正态分布)的Kaiming初始化权重参数
        nn.init.kaiming_normal_(self.layer2.weight, nonlinearity='relu')
        nn.init.zeros_(self.layer2.bias)

    def forward(self, x):
        # 前向传播的计算过程
        x = torch.relu(self.layer1(x))  # 第一层使用Sigmoid激活函数
        x = torch.relu(self.layer2(x))  # 第二层使用ReLU激活函数
        x = torch.softmax(self.layer3(x), dim=-1) # 输出层有多个输出, 可以使用Softmax激活函数. 其中dim=-1表示对最后一个维度进行Softmax计算, 也就是对每个样本的输出向量进行Softmax计算.

        # 最后返回预测值
        return x

定义好模型之后, 创建模型对象, 查看模型结构信息, 测试前向传播

前向传播结果的形状: (样本数量或者batch_size, 输出层的输出维度大小)

In [24]:
# 创建模型对象
model = SimpleNet()
# 打印模型结构信息
# summary(model, input_size=(5, 3))  # 输入维度为 (5, 3), 本来应该表示5样本3特征, 但是summary包在这里会理解为"单个样本的形状为(5, 3)", 于是输出的信息中会显示输出形状为"[-1, 5, x]", 在前面补上了批次维度, 表示样本/批次数量不确定, 但每个样本/中间值的形状是[5, x]

summary(model, input_size=(3, ), batch_size=5) # 直接指定batch_size就不容易混淆了, 这里的输入维度为 (3, ), 表示每个样本有3个特征, batch_size=5表示每次输入5个样本

# print(model)

# 测试前向传播
test = torch.randn(5, 3) # 创建一个5个样本, 每个样本有3个特征的随机输入张量

with torch.no_grad(): # 测试前向传播时不需要计算梯度, torch.no_grad()禁止底层计算图的构建, 节省内存和计算资源
    output = model(test)
print(output)

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Linear-1                     [5, 3]              12
            Linear-2                     [5, 2]               8
            Linear-3                     [5, 2]               6
Total params: 26
Trainable params: 26
Non-trainable params: 0
----------------------------------------------------------------
Input size (MB): 0.00
Forward/backward pass size (MB): 0.00
Params size (MB): 0.00
Estimated Total Size (MB): 0.00
----------------------------------------------------------------
tensor([[0.4751, 0.5249],
        [0.4772, 0.5228],
        [0.4838, 0.5162],
        [0.4799, 0.5201],
        [0.4666, 0.5334]])
